In [16]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [17]:
from rag_core import setup_ssl

setup_ssl()

In [18]:
load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "rag-llm-index")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")
PDF_DIR = Path(os.getenv("PDF_DIR", "./data"))
MIN_RELEVANCE_SCORE = float(os.getenv("MIN_RELEVANCE_SCORE", "0.75"))

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY in environment or .env")

if not PINECONE_API_KEY:
    raise ValueError("Missing PINECONE_API_KEY in environment or .env")

if not PDF_DIR.exists():
    raise FileNotFoundError(f"PDF directory not found: {PDF_DIR.resolve()}")

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in: {PDF_DIR.resolve()}")

In [19]:
all_pages = []

for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    all_pages.extend(pages)
    print(f"Loaded {len(pages)} pages from {pdf_path.name}")

print(f"Total pages loaded: {len(all_pages)}")

Loaded 40 pages from 778071.pdf
Loaded 12 pages from DisabledRights.pdf
Loaded 12 pages from EzrahVatik.pdf
Loaded 36 pages from terror_victims_rights.pdf
Loaded 16 pages from yoledet.pdf
Total pages loaded: 116


In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(all_pages)
print(f"Created {len(chunks)} chunks")

Created 230 chunks


In [21]:
pc = Pinecone(api_key=PINECONE_API_KEY)
spec = ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=3072,
        metric="dotproduct",
        spec=spec,
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
index = pc.Index(PINECONE_INDEX_NAME)
index.delete(delete_all=True)
print("Cleared existing vectors from Pinecone index")

Cleared existing vectors from Pinecone index


In [22]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=GEMINI_API_KEY
)

text_chunks = [chunk.page_content for chunk in chunks]
metadata = [
    {
        "chunk_id": i,
        "source": Path(chunks[i].metadata.get("source", "unknown")).name,
        "text": text_chunks[i],
    }
    for i in range(len(text_chunks))
]

embeddings = []
for i, text in enumerate(text_chunks):
    embedding = embeddings_model.embed_query(text)
    embeddings.append(embedding)
    print(f"Embedded {i+1}/{len(text_chunks)}")
    time.sleep(0.7)

Embedded 1/230
Embedded 2/230
Embedded 3/230
Embedded 4/230
Embedded 5/230
Embedded 6/230
Embedded 7/230
Embedded 8/230
Embedded 9/230
Embedded 10/230
Embedded 11/230
Embedded 12/230
Embedded 13/230
Embedded 14/230
Embedded 15/230
Embedded 16/230
Embedded 17/230
Embedded 18/230
Embedded 19/230
Embedded 20/230
Embedded 21/230
Embedded 22/230
Embedded 23/230
Embedded 24/230
Embedded 25/230
Embedded 26/230
Embedded 27/230
Embedded 28/230
Embedded 29/230
Embedded 30/230
Embedded 31/230
Embedded 32/230
Embedded 33/230
Embedded 34/230
Embedded 35/230
Embedded 36/230
Embedded 37/230
Embedded 38/230
Embedded 39/230
Embedded 40/230
Embedded 41/230
Embedded 42/230
Embedded 43/230
Embedded 44/230
Embedded 45/230
Embedded 46/230
Embedded 47/230
Embedded 48/230
Embedded 49/230
Embedded 50/230
Embedded 51/230
Embedded 52/230
Embedded 53/230
Embedded 54/230
Embedded 55/230
Embedded 56/230
Embedded 57/230
Embedded 58/230
Embedded 59/230
Embedded 60/230
Embedded 61/230
Embedded 62/230
Embedded 63/230
E

# שמירה ב-Pinecone

In [23]:
def save_embeddings_to_pinecone(embeddings, metadata, namespace=None, batch_size=50):
    total_uploaded = 0
    try:
        for start in range(0, len(embeddings), batch_size):
            end = min(start + batch_size, len(embeddings))
            vectors = [
                {"id": f"id-{i}", "values": embeddings[i], "metadata": metadata[i]}
                for i in range(start, end)
            ]
            if namespace:
                index.upsert(vectors=vectors, namespace=namespace)
            else:
                index.upsert(vectors=vectors)
            total_uploaded += len(vectors)
            print(f"Uploaded batch {start}-{end - 1}")
        return total_uploaded
    except Exception as exc:
        raise RuntimeError("Failed to upsert vectors to Pinecone.") from exc

In [24]:
uploaded = save_embeddings_to_pinecone(embeddings, metadata)
print(f"Uploaded {uploaded} vectors to Pinecone")

Uploaded batch 0-49
Uploaded batch 50-99
Uploaded batch 100-149
Uploaded batch 150-199
Uploaded batch 200-229
Uploaded 230 vectors to Pinecone


In [25]:
def retrieve(query: str, top_k: int = 3, min_score: float = MIN_RELEVANCE_SCORE):
    query_vector = embeddings_model.embed_query(query)

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
    )

    matches = [m for m in results["matches"] if m["score"] >= min_score]

    print(f"\n🔍 Query: {query}\n")
    if not matches:
        print(f"לא נמצאו קטעים רלוונטיים (סף מינימלי: {min_score:.2f}).")
        print()
        return {"matches": []}

    for i, match in enumerate(matches):
        print(f"--- Match {i+1} (score: {match['score']:.3f}) ---")
        print(f"Source: {match['metadata']['source']}")
        print(f"Chunk ID: {match['metadata']['chunk_id']}")
        print(f"Text: {match['metadata']['text']}")
        print()

    return {"matches": matches}

In [26]:
questions = [
    "מה סכום ההשתתפות בהוצאות הקבורה, ואיך מגישים בקשה לתשלום?",
    "מהם תנאי הזכאות לקצבת נכות כללית?",
    "מה הסכום המקסימלי להכנסה מעבודה שעדיין מאפשר לקבל את קצבת אזרח ותיק במלואה?",
    "אם נכחתי באירוע איבה וקיבלתי סירוב לאישור הכרה, האם אני עדיין זכאי לקבל טיפול רפואי ראשוני והחזר על אביזרים רפואיים שניזוקו?",
   "מהו סכום מענק הלידה שאקבל עבור ילד שלישי, ולאיזה חשבון בנק הוא משולם?",
]

for q in questions:
    retrieve(q)
    time.sleep(1)


🔍 Query: מה סכום ההשתתפות בהוצאות הקבורה, ואיך מגישים בקשה לתשלום?

--- Match 1 (score: 0.820) ---
Source: 778071.pdf
Chunk ID: 12
Text: 8
כל הזכויות בביטוח הלאומי
השתתפות בהוצאות הקבורה
תשלום לבן משפחה של החלל שנשא בהוצאות הקבורה, כגון מצבה, העברת 
הגופה ממקום אחד למשנהו בישראל, תשלום לרכב ליווי.
 ₪13,316סכום ההשתתפות
כיצד מקבלים את התשלום?
בן המשפחה צריך למלא הצהרה על סכום ההוצאות ולשלוח אותה ישירות 
.לסניף המטפל, באמצעות שירות שליחת מסמכים באתר
מענקי הנצחה
משפחות יקרות, אנו יודעים שחשוב לכן להנציח את יקירכן, לכן הביטוח הלאומי 
 נותן לכן מענקים ותמיכה בפרויקטים של הנצחה באופן הבא:
מענק להנצחה פרטית

--- Match 2 (score: 0.789) ---
Source: 778071.pdf
Chunk ID: 16
Text: ₪1,083סכום ההשתתפות
באופן אוטומטי בתגמול של חודש איך מקבלים את התשלום? 
הפטירה הלועזי.
- יש להגיש בקשה להשתתפות בהוצאות במקרים שבהם לא משולם תגמול 
אזכרה למחלקת נפגעי האיבה בסניף המטפל בכם. אפשר לשלוח את הבקשה 
גם באמצעות שירות שליחת מסמכים באתר.
- התשלום יועבר לקרוב אם לחלל אין שאירים, כלומר אין לו בת זוג או ילדים 
משפ

In [27]:
from google import genai
from rag_core import generate_answer

client = genai.Client(api_key=GEMINI_API_KEY)

In [28]:
result = generate_answer(
    "מה סכום ההשתתפות בהוצאות הקבורה, ואיך מגישים בקשה לתשלום?",
    client, index, embeddings_model, MIN_RELEVANCE_SCORE,
)
print(f"🤖 Answer:\n{result['answer']}")
if result["sources"]:
    for src in result["sources"]:
        print(f"\n--- Source: {src['source']} (score: {src['score']:.3f}) ---\n{src['text']}")


🔍 Query: מה סכום ההשתתפות בהוצאות הקבורה, ואיך מגישים בקשה לתשלום?

📄 Context used:
8
כל הזכויות בביטוח הלאומי
השתתפות בהוצאות הקבורה
תשלום לבן משפחה של החלל שנשא בהוצאות הקבורה, כגון מצבה, העברת 
הגופה ממקום אחד למשנהו בישראל, תשלום לרכב ליווי.
 ₪13,316סכום ההשתתפות
כיצד מקבלים את התשלום?
בן המשפחה צריך למלא הצהרה על סכום ההוצאות ולשלוח אותה ישירות 
.לסניף המטפל, באמצעות שירות שלי...

🤖 Answer:
סכום ההשתתפות בהוצאות הקבורה הוא 13,316 ש"ח.
כדי לקבל את התשלום, בן המשפחה צריך למלא הצהרה על סכום ההוצאות ולשלוח אותה ישירות לסניף המטפל, באמצעות שירות שליחת מסמכים באתר.
